# ScholarGraph SG-002 — compute smoke test

Runs a deterministic 1,000-document encoding pilot on CPU or CUDA and writes `artifacts/reports/compute_pilot.json`. The built-in texts are a **synthetic compute fixture**, never evaluation data. Set `SCHOLARGRAPH_PILOT_JSONL` to a JSONL corpus with `title`/`abstract` fields to use real documents. The default MiniLM model is for infrastructure measurement only; SG-017 chooses the scientific retriever.

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Settings → Accelerator → GPU. Run all cells from the project root or set `SCHOLARGRAPH_PROJECT_ROOT`.

In [ ]:
import importlib.util, subprocess, sys

required = {'sentence_transformers': 'sentence-transformers>=3.0,<6', 'psutil': 'psutil>=5.9'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready:', ', '.join(required))

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import psutil
import sentence_transformers
import torch
import transformers
from sentence_transformers import SentenceTransformer

TASK_ID = 'SG-002'
MODEL_ID = os.getenv('SCHOLARGRAPH_PILOT_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')
MODEL_REVISION_REQUESTED = os.getenv('SCHOLARGRAPH_PILOT_REVISION', 'main')
N_DOCS = 1000
MAX_SEQ_LENGTH = int(os.getenv('SCHOLARGRAPH_MAX_SEQ_LENGTH', '256'))
PROJECT_ROOT = Path(os.getenv('SCHOLARGRAPH_PROJECT_ROOT', Path.cwd())).resolve()
INPUT_JSONL = os.getenv('SCHOLARGRAPH_PILOT_JSONL')
REPORT_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / 'compute_pilot.json'
EMBEDDINGS_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / 'compute_pilot_embeddings.npy'
CACHE_DIR = PROJECT_ROOT / 'artifacts' / 'cache' / 'huggingface'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
default_batch = 32 if device == 'cuda' else 8
BATCH_SIZE = int(os.getenv('SCHOLARGRAPH_BATCH_SIZE', str(default_batch)))
print({'project_root': str(PROJECT_ROOT), 'device': device, 'batch_size': BATCH_SIZE, 'model': MODEL_ID})

In [ ]:
def load_texts():
    if INPUT_JSONL:
        path = Path(INPUT_JSONL).expanduser().resolve()
        rows = []
        with path.open('r', encoding='utf-8') as handle:
            for line in handle:
                if not line.strip():
                    continue
                row = json.loads(line)
                text = ' '.join(str(row.get(k) or '').strip() for k in ('title', 'abstract')).strip()
                if text:
                    rows.append(text)
                if len(rows) == N_DOCS:
                    break
        if len(rows) != N_DOCS:
            raise ValueError(f'Need exactly {N_DOCS} non-empty texts, got {len(rows)} from {path}')
        return rows, {'kind': 'jsonl', 'path': str(path), 'synthetic': False}

    topics = [
        'information retrieval and relevance ranking',
        'citation graph analysis and personalized pagerank',
        'scientific document representation and dense retrieval',
        'cross lingual search for Vietnamese research questions',
        'temporal bias and exposure in scholarly recommendation',
    ]
    clauses = [
        'We evaluate a reproducible baseline with controlled splits and report limitations.',
        'The study compares lexical semantic and graph signals without using test labels as features.',
        'Experiments record model revision data provenance runtime memory and per query metrics.',
    ]
    rows = []
    for i in range(N_DOCS):
        repeat = 2 + (i % 5)
        rows.append(f'Paper {i}: {topics[i % len(topics)]}. ' + ' '.join(clauses[(i+j) % len(clauses)] for j in range(repeat)))
    return rows, {'kind': 'deterministic_synthetic_compute_fixture', 'seed': 0, 'synthetic': True}

texts, input_info = load_texts()
input_sha256 = hashlib.sha256('\n'.join(texts).encode('utf-8')).hexdigest()
lengths = np.array([len(t.split()) for t in texts])
print({'documents': len(texts), 'input': input_info, 'word_length_p50': float(np.median(lengths)), 'word_length_p95': float(np.percentile(lengths, 95)), 'sha256': input_sha256})

In [ ]:
def nvidia_smi_snapshot():
    exe = shutil.which('nvidia-smi')
    if not exe:
        return {'available': False, 'rows': []}
    query = 'name,driver_version,memory.total,memory.used,memory.free,utilization.gpu,temperature.gpu'
    proc = subprocess.run([exe, f'--query-gpu={query}', '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=20)
    return {'available': proc.returncode == 0, 'returncode': proc.returncode, 'rows': [r.strip() for r in proc.stdout.splitlines() if r.strip()], 'stderr': proc.stderr.strip()}

process = psutil.Process()
ram = psutil.virtual_memory()
disk = shutil.disk_usage(PROJECT_ROOT)
before_gpu = nvidia_smi_snapshot()
rss_before = process.memory_info().rss

if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

load_started = time.perf_counter()
model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION_REQUESTED, device=device, cache_folder=str(CACHE_DIR))
model.max_seq_length = MAX_SEQ_LENGTH
model_load_seconds = time.perf_counter() - load_started
resolved_revision = getattr(getattr(model[0], 'auto_model', None).config, '_commit_hash', None)

_ = model.encode(texts[:16], batch_size=min(BATCH_SIZE, 16), convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=True)
if device == 'cuda':
    torch.cuda.synchronize()
encode_started = time.perf_counter()
embeddings = model.encode(texts, batch_size=BATCH_SIZE, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
if device == 'cuda':
    torch.cuda.synchronize()
encode_seconds = time.perf_counter() - encode_started
rss_after = process.memory_info().rss
after_gpu = nvidia_smi_snapshot()

if embeddings.shape[0] != N_DOCS or not np.isfinite(embeddings).all():
    raise AssertionError(f'Invalid embeddings: shape={embeddings.shape}, finite={np.isfinite(embeddings).all()}')
norms = np.linalg.norm(embeddings, axis=1)
if not np.allclose(norms, 1.0, atol=1e-4):
    raise AssertionError(f'Normalization failed: min={norms.min()}, max={norms.max()}')
np.save(EMBEDDINGS_PATH, embeddings, allow_pickle=False)
embedding_sha256 = hashlib.sha256(EMBEDDINGS_PATH.read_bytes()).hexdigest()

docs_per_second = N_DOCS / encode_seconds
estimate = lambda n: {'seconds_raw': n / docs_per_second, 'seconds_with_25pct_overhead': n / docs_per_second * 1.25}
cuda_props = torch.cuda.get_device_properties(0) if device == 'cuda' else None
peak_allocated = torch.cuda.max_memory_allocated() if device == 'cuda' else None
peak_reserved = torch.cuda.max_memory_reserved() if device == 'cuda' else None

report = {
    'schema_version': 1,
    'run_id': f"sg002-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}",
    'task_id': TASK_ID,
    'status': 'completed_gpu' if device == 'cuda' else 'completed_local_cpu_gpu_pending',
    'started_and_finished_in_notebook': True,
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'purpose': 'Compute throughput and memory pilot only; not retrieval quality evidence.',
    'input': {**input_info, 'documents': N_DOCS, 'sha256': input_sha256, 'word_length_p50': float(np.median(lengths)), 'word_length_p95': float(np.percentile(lengths, 95))},
    'model': {'id': MODEL_ID, 'requested_revision': MODEL_REVISION_REQUESTED, 'resolved_revision': resolved_revision, 'selection_status': 'smoke_only_not_final_sg017', 'max_seq_length': MAX_SEQ_LENGTH},
    'environment': {
        'platform': platform.platform(), 'python': platform.python_version(),
        'torch': torch.__version__, 'torch_cuda_build': torch.version.cuda,
        'transformers': transformers.__version__, 'sentence_transformers': sentence_transformers.__version__,
        'logical_cpu_count': psutil.cpu_count(logical=True), 'physical_cpu_count': psutil.cpu_count(logical=False),
        'ram_total_bytes': ram.total, 'ram_available_before_bytes': ram.available,
        'project_disk_total_bytes': disk.total, 'project_disk_free_before_bytes': disk.free,
        'device': device, 'cuda_available': torch.cuda.is_available(),
        'gpu_name': cuda_props.name if cuda_props else None,
        'gpu_total_memory_bytes': cuda_props.total_memory if cuda_props else None,
        'nvidia_smi_before': before_gpu, 'nvidia_smi_after': after_gpu,
    },
    'measurement': {
        'batch_size': BATCH_SIZE, 'warmup_documents': 16,
        'model_load_seconds': model_load_seconds, 'encode_seconds': encode_seconds,
        'documents_per_second': docs_per_second,
        'process_rss_before_bytes': rss_before, 'process_rss_after_bytes': rss_after,
        'process_rss_delta_bytes': rss_after - rss_before,
        'gpu_peak_allocated_bytes': peak_allocated, 'gpu_peak_reserved_bytes': peak_reserved,
        'embedding_shape': list(embeddings.shape), 'embedding_dtype': str(embeddings.dtype),
        'embedding_file_bytes': EMBEDDINGS_PATH.stat().st_size, 'embedding_sha256': embedding_sha256,
        'norm_min': float(norms.min()), 'norm_max': float(norms.max()),
    },
    'estimates_from_this_run': {'10000_documents': estimate(10000), '20000_documents': estimate(20000), '50000_documents': estimate(50000)},
    'cloud_gpu': {'provider': None, 'status': 'measured_in_this_run' if device == 'cuda' else 'not_run', 'account_quota': 'not_verified'},
    'limitations': [
        'Synthetic fixture measures compute behavior only and cannot support retrieval-quality claims.' if input_info['synthetic'] else 'Input is a user-supplied sample; corpus representativeness must be audited separately.',
        'The default smoke model is not the final scientific retriever; rerun after SG-017 with the selected model.',
        'Free Colab/Kaggle hardware and quota are dynamic; report only the accelerator actually observed.',
    ] + ([] if device == 'cuda' else ['GPU throughput and peak VRAM remain unverified; rerun this notebook on an enabled Colab/Kaggle GPU.']),
}
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'status': report['status'], 'report': str(REPORT_PATH), 'device': device, 'docs_per_second': docs_per_second, 'encode_seconds': encode_seconds, 'embedding_shape': embeddings.shape, 'gpu_peak_allocated_bytes': peak_allocated}, indent=2))

In [ ]:
saved = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert saved['input']['documents'] == 1000
assert saved['measurement']['documents_per_second'] > 0
assert saved['measurement']['embedding_shape'][0] == 1000
print('VALID:', saved['run_id'], saved['status'])
print('10K estimate with overhead (minutes):', round(saved['estimates_from_this_run']['10000_documents']['seconds_with_25pct_overhead'] / 60, 2))
print('20K estimate with overhead (minutes):', round(saved['estimates_from_this_run']['20000_documents']['seconds_with_25pct_overhead'] / 60, 2))